# Notebook 02: AgentCore Runtime Setup

## Learning Objectives
- Configure AgentCore Runtime for travel agent
- Create basic conversational agent using Strands
- Implement multi-turn dialogue patterns
- Test conversation flow and state management

## Prerequisites
- Completed Notebook 01 (Foundation)
- AWS environment validated
- API keys configured

## Step 1: Connect to your AWS environment

In [1]:
import os

os.environ['AWS_REGION'] = 'us-east-1'

# APPROACH A: Use credentials
os.environ['AWS_ACCESS_KEY_ID'] = 'YOUR_AWS_ACCESS_KEY_ID'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'YOUR_AWS_SECRET_ACCESS_KEY'
# os.environ['AWS_SESSION_TOKEN'] = "your_session_token"

# APPROACH B: Use AWS SSO profile
#os.environ['AWS_PROFILE'] = 'your_profile'
# Remove any existing credential env vars to force profile usage
#for key in ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_SESSION_TOKEN']:
#    os.environ.pop(key, None)

os.environ['AWS_REGION'] = 'us-east-1'

print("✅ AWS Profile set. Please restart kernel and run all cells.")

✅ AWS Profile set. Please restart kernel and run all cells.


In [1]:
import os
from dotenv import load_dotenv
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel

# Load environment variables
load_dotenv()

print("✅ AgentCore Runtime imports successful")

✅ AgentCore Runtime imports successful


## Step 2: Create Basic Travel Agent Tools

In [2]:
# Define basic travel tools for our agent
@tool
def get_travel_preferences():
    """Get user's travel preferences from memory"""
    # Mock implementation - will be enhanced in Memory notebook
    return {
        "hotel_type": "mid-range",
        "food_preference": "vegetarian",
        "budget_range": "moderate"
    }

@tool
def calculate_budget(total_budget: int, days: int):
    """Calculate daily budget allocation for travel"""
    # Basic budget breakdown
    daily_budget = total_budget / days
    allocation = {
        "flights": total_budget * 0.24,  # 24%
        "hotels": total_budget * 0.36,   # 36%
        "food": total_budget * 0.20,     # 20%
        "activities": total_budget * 0.16, # 16%
        "buffer": total_budget * 0.04     # 4%
    }
    return {
        "daily_budget": daily_budget,
        "allocation": allocation
    }

@tool
def get_destination_info(destination: str):
    """Get basic information about a travel destination"""
    # Mock implementation - will be enhanced with real APIs
    destinations = {
        "rome": {
            "country": "Italy",
            "currency": "EUR",
            "language": "Italian",
            "attractions": ["Colosseum", "Vatican", "Trevi Fountain"]
        },
        "florence": {
            "country": "Italy",
            "currency": "EUR",
            "language": "Italian",
            "attractions": ["Uffizi Gallery", "Ponte Vecchio", "Duomo"]
        },
        "venice": {
            "country": "Italy",
            "currency": "EUR",
            "language": "Italian",
            "attractions": ["St. Mark's Square", "Grand Canal", "Doge's Palace"]
        }
    }
    return destinations.get(destination.lower(), {"error": "Destination not found"})

print("✅ Travel agent tools defined")

✅ Travel agent tools defined


## Step 3: Create Travel Agent with Bedrock Model

In [9]:
# Initialize Bedrock model for the agent
#model_id = "anthropic.claude-3-haiku-202410307-v1:0"
#model_id = "anthropic.claude-3-haiku-20240307-v1:0"
model_id = "amazon.nova-lite-v1:0"
model = BedrockModel(model_id=model_id)

# Create the travel agent with system prompt
system_prompt = """
You are an AI Travel Companion specializing in planning trips to Italy. 
Your expertise includes:
- Flight and hotel recommendations
- Budget optimization and allocation
- Destination information and attractions
- Personalized recommendations based on user preferences

Always ask clarifying questions to better understand the user's needs.
Be helpful, friendly, and provide detailed explanations for your recommendations.
Remember user preferences and reference them in future interactions.
"""

travel_agent = Agent(
    model=model,
    tools=[get_travel_preferences, calculate_budget, get_destination_info],
    system_prompt=system_prompt
)

print("✅ Travel agent created with Bedrock model")

✅ Travel agent created with Bedrock model


## Step 4: Test Local Agent Functionality

In [10]:
# Test the agent locally before deploying to runtime
def test_travel_agent(user_input):
    """Test function for local agent invocation"""
    print(f"User: {user_input}")
    response = travel_agent(user_input)
    agent_response = response.message['content'][0]['text']
    print(f"Agent: {agent_response}")
    return agent_response

# Test basic functionality
print("🧪 Testing Travel Agent Locally")
print("=" * 50)

test_travel_agent("Hi, I want to plan a trip to Italy")

🧪 Testing Travel Agent Locally
User: Hi, I want to plan a trip to Italy
<thinking>I need to gather more information about the user's preferences and requirements before I can provide specific recommendations. I'll start by asking some clarifying questions.</thinking>

Hi! That's fantastic! Italy is a beautiful country with so much to offer. To help you plan the perfect trip, I'll need a bit more information:

1. **Destinations**: Are there any specific cities or regions in Italy you're interested in visiting?
2. **Duration**: How many days are you planning to stay in Italy?
3. **Budget**: What is your total budget for the trip?
4. **Travel Style**: Do you prefer a relaxed pace, or are you looking to explore multiple cities in a short time?
5. **Accommodation Preferences**: Do you have any preferences for accommodation types (e.g., hotels, hostels, Airbnb)?
6. **Activities**: Are there any specific activities or attractions you want to prioritize (e.g., historical sites, beaches, food t

"<thinking>I need to gather more information about the user's preferences and requirements before I can provide specific recommendations. I'll start by asking some clarifying questions.</thinking>\n\nHi! That's fantastic! Italy is a beautiful country with so much to offer. To help you plan the perfect trip, I'll need a bit more information:\n\n1. **Destinations**: Are there any specific cities or regions in Italy you're interested in visiting?\n2. **Duration**: How many days are you planning to stay in Italy?\n3. **Budget**: What is your total budget for the trip?\n4. **Travel Style**: Do you prefer a relaxed pace, or are you looking to explore multiple cities in a short time?\n5. **Accommodation Preferences**: Do you have any preferences for accommodation types (e.g., hotels, hostels, Airbnb)?\n6. **Activities**: Are there any specific activities or attractions you want to prioritize (e.g., historical sites, beaches, food tours)?\n\nFeel free to provide as much or as little detail as 

In [11]:
# Test budget calculation
test_travel_agent("I have a budget of $5000 for a 10-day trip. How should I allocate it?")

User: I have a budget of $5000 for a 10-day trip. How should I allocate it?
<thinking>The user has provided a total budget and the duration of the trip. I can now use the "calculate_budget" tool to help allocate the budget effectively. I'll also ask for some additional preferences to ensure the allocation is tailored to the user's needs.</thinking>

Great, let's start by allocating your budget. Here's a rough estimate of how you might break it down:

- **Flights**: Approximately 20-25% of the total budget.
- **Accommodation**: Approximately 30-35% of the total budget.
- **Food and Dining**: Approximately 20-25% of the total budget.
- **Transportation**: Approximately 10-15% of the total budget.
- **Attractions and Activities**: Approximately 10-15% of the total budget.

Let's calculate the exact amounts based on your budget of $5000 for a 10-day trip.


Tool #1: calculate_budget
<thinking>The budget allocation has been calculated. Now, I'll provide the user with a detailed breakdown an

"<thinking>The budget allocation has been calculated. Now, I'll provide the user with a detailed breakdown and some recommendations based on this allocation.</thinking>\n\nHere's a breakdown of your budget allocation for a 10-day trip to Italy with a total budget of $5000:\n\n- **Daily Budget**: $500 per day\n- **Flights**: $1200\n- **Hotels**: $1800\n- **Food**: $1000\n- **Activities**: $800\n- **Buffer**: $200\n\n### Recommendations:\n\n1. **Flights**:\n   - Allocate around $1200 for flights. This should cover round-trip economy class tickets from your home country to Italy.\n\n2. **Accommodation**:\n   - With $1800, you can stay in comfortable hotels or well-rated hostels. Mid-range hotels typically cost around $100-$150 per night, which should fit well within your budget.\n\n3. **Food**:\n   - Budget $1000 for food. Italy is famous for its cuisine, and you can enjoy delicious meals for a reasonable price. Allocate around $100 per day for dining.\n\n4. **Activities**:\n   - With $80

In [12]:
# Test destination information
test_travel_agent("Tell me about Rome and what I should see there")

User: Tell me about Rome and what I should see there
<thinking>The user has expressed interest in Rome. I'll use the "get_destination_info" tool to gather basic information about Rome and its attractions. Then, I'll provide a list of must-see sites and activities.</thinking>


Tool #2: get_destination_info
<thinking>I have the basic information about Rome. Now, I'll provide a detailed list of must-see attractions and activities in Rome based on the user's budget and preferences.</thinking>

### Rome, Italy

**Currency**: EUR (Euro)
**Language**: Italian

### Must-See Attractions in Rome:

1. **Colosseum**:
   - **Description**: An iconic ancient amphitheater where gladiators once fought.
   - **Cost**: Approximately €16 for an adult ticket.
   - **Recommendation**: Consider a guided tour to learn about its history and significance.

2. **Vatican City**:
   - **Description**: The smallest independent state in the world, home to St. Peter's Basilica, the Sistine Chapel, and the Vatican M

"<thinking>I have the basic information about Rome. Now, I'll provide a detailed list of must-see attractions and activities in Rome based on the user's budget and preferences.</thinking>\n\n### Rome, Italy\n\n**Currency**: EUR (Euro)\n**Language**: Italian\n\n### Must-See Attractions in Rome:\n\n1. **Colosseum**:\n   - **Description**: An iconic ancient amphitheater where gladiators once fought.\n   - **Cost**: Approximately €16 for an adult ticket.\n   - **Recommendation**: Consider a guided tour to learn about its history and significance.\n\n2. **Vatican City**:\n   - **Description**: The smallest independent state in the world, home to St. Peter's Basilica, the Sistine Chapel, and the Vatican Museums.\n   - **Cost**: €17 for the Vatican Museums and Sistine Chapel combined ticket.\n   - **Recommendation**: Visit early in the morning to avoid crowds and book tickets online in advance.\n\n3. **Trevi Fountain**:\n   - **Description**: A stunning Baroque fountain known for its beauty a

## Step 5: Prepare Agent for AgentCore Runtime

In [13]:
%%writefile ../backend/runtime/simple_agent/travel_agent.py
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
import json

# Initialize AgentCore Runtime App
app = BedrockAgentCoreApp()

# Define travel tools
@tool
def get_travel_preferences():
    """Get user's travel preferences from memory"""
    return {
        "hotel_type": "mid-range",
        "food_preference": "vegetarian",
        "budget_range": "moderate"
    }

@tool
def calculate_budget(total_budget: int, days: int):
    """Calculate daily budget allocation for travel"""
    daily_budget = total_budget / days
    allocation = {
        "flights": total_budget * 0.24,
        "hotels": total_budget * 0.36,
        "food": total_budget * 0.20,
        "activities": total_budget * 0.16,
        "buffer": total_budget * 0.04
    }
    return {
        "daily_budget": daily_budget,
        "allocation": allocation
    }

@tool
def get_destination_info(destination: str):
    """Get basic information about a travel destination"""
    destinations = {
        "rome": {
            "country": "Italy",
            "currency": "EUR",
            "language": "Italian",
            "attractions": ["Colosseum", "Vatican", "Trevi Fountain"]
        },
        "florence": {
            "country": "Italy",
            "currency": "EUR",
            "language": "Italian",
            "attractions": ["Uffizi Gallery", "Ponte Vecchio", "Duomo"]
        },
        "venice": {
            "country": "Italy",
            "currency": "EUR",
            "language": "Italian",
            "attractions": ["St. Mark's Square", "Grand Canal", "Doge's Palace"]
        }
    }
    return destinations.get(destination.lower(), {"error": "Destination not found"})

# Initialize model and agent
model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
model = BedrockModel(model_id=model_id)

system_prompt = """
You are an AI Travel Companion specializing in planning trips to Italy. 
Your expertise includes:
- Flight and hotel recommendations
- Budget optimization and allocation
- Destination information and attractions
- Personalized recommendations based on user preferences

Always ask clarifying questions to better understand the user's needs.
Be helpful, friendly, and provide detailed explanations for your recommendations.
Remember user preferences and reference them in future interactions.
"""

travel_agent = Agent(
    model=model,
    tools=[get_travel_preferences, calculate_budget, get_destination_info],
    system_prompt=system_prompt
)

@app.entrypoint
def invoke_travel_agent(payload):
    """AgentCore Runtime entrypoint for travel agent"""
    user_input = payload.get("prompt", "")
    print(f"User input: {user_input}")
    
    response = travel_agent(user_input)
    agent_response = response.message['content'][0]['text']
    
    return agent_response

if __name__ == "__main__":
    app.run()

Overwriting ../backend/runtime/simple_agent/travel_agent.py


In [14]:
%%writefile ../backend/runtime/simple_agent/requirements.txt
# Core Amazon Bedrock AgentCore dependencies
bedrock-agentcore>=1.0.5
bedrock-agentcore-starter-toolkit>=0.1.27

# AWS SDK and utilities
boto3>=1.40.62
python-dotenv>=1.2.1

# Agent framework
strands-agents>=1.14.0

Overwriting ../backend/runtime/simple_agent/requirements.txt


## Step 6: Deploy to AgentCore Runtime

In [15]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import os
from pathlib import Path

# Initialize runtime deployment
boto_session = Session()
region = boto_session.region_name or "us-east-1"

# Change to project root for AgentCore configuration
original_dir = os.getcwd()
os.chdir('../backend/runtime/simple_agent')
project_root = os.getcwd()

agentcore_runtime = Runtime()
agent_name = "travel_companion_basic"

print(f"🚀 Configuring AgentCore Runtime deployment...")
print(f"Agent Name: {agent_name}")
print(f"Region: {region}")
print(f"Project Root: {project_root}")

# Configure with paths relative to project root
configure_response = agentcore_runtime.configure(
    entrypoint="travel_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    container_runtime=None  # Force CodeBuild for cross-platform build
)

print("✅ Runtime configuration complete")
print("💡 Using CodeBuild for cross-platform ARM64 deployment")
configure_response

Entrypoint parsed: file=C:\Users\srinivas.susarapu\mastering-amazon-bedrock-agentcore\capstone_project\backend\runtime\simple_agent\travel_agent.py, bedrock_agentcore_name=travel_agent
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: travel_companion_basic


🚀 Configuring AgentCore Runtime deployment...
Agent Name: travel_companion_basic
Region: us-east-1
Project Root: c:\Users\srinivas.susarapu\mastering-amazon-bedrock-agentcore\capstone_project\backend\runtime\simple_agent


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


⚠️ Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64', so local builds
won't work.
Please use default launch command which will do a remote cross-platform build using code build.For deployment other
options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

Generated .dockerignore
Generated Dockerfile: Dockerfile
Generated .dockerignore: c:\Users\srinivas.susarapu\mastering-amazon-bedrock-agentcore\capstone_project\backend\runtime\simple_agent\.dockerignore
Setting 'travel_companion_basic' as default agent
Bedrock AgentCore configured: c:\Users\srinivas.susarapu\mastering-amazon-bedrock-agentcore\capstone_project\backend\runtime\simple_agent\.bedrock_agentcore.yaml


✅ Runtime configuration complete
💡 Using CodeBuild for cross-platform ARM64 deployment


ConfigureResult(config_path=WindowsPath('c:/Users/srinivas.susarapu/mastering-amazon-bedrock-agentcore/capstone_project/backend/runtime/simple_agent/.bedrock_agentcore.yaml'), dockerfile_path=WindowsPath('c:/Users/srinivas.susarapu/mastering-amazon-bedrock-agentcore/capstone_project/backend/runtime/simple_agent/Dockerfile'), dockerignore_path=WindowsPath('c:/Users/srinivas.susarapu/mastering-amazon-bedrock-agentcore/capstone_project/backend/runtime/simple_agent/.dockerignore'), runtime='None', runtime_type=None, region='us-east-1', account_id='790402872574', execution_role=None, ecr_repository=None, auto_create_ecr=True, s3_path=None, auto_create_s3=False, memory_id=None, network_mode='PUBLIC', network_subnets=None, network_security_groups=None, network_vpc_id=None)

In [16]:
# Launch the agent to AgentCore Runtime
print("🚀 Launching agent to AgentCore Runtime...")
print("This may take 5-10 minutes...")

launch_result = agentcore_runtime.launch()
print("✅ Launch initiated")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"ECR URI: {launch_result.ecr_uri}")

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'travel_companion_basic' to account 790402872574 (us-east-1)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: travel_companion_basic


🚀 Launching agent to AgentCore Runtime...
This may take 5-10 minutes...
Repository doesn't exist, creating new ECR repository: bedrock-agentcore-travel_companion_basic


ECR repository available: 790402872574.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-travel_companion_basic
Getting or creating execution role for agent: travel_companion_basic
Using AWS region: us-east-1, account ID: 790402872574
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-346339bc6e
Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-346339bc6e
Starting execution role creation process for agent: travel_companion_basic
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-us-east-1-346339bc6e
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-346339bc6e
✓ Role created: arn:aws:iam::790402872574:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-346339bc6e
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-travel_companion_basic
Role creation complete and ready for use with Bedrock AgentCore
Execution role available: arn:aws:iam::790402872574:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-346339bc6e
Preparing C

✅ Launch initiated
Agent ARN: arn:aws:bedrock-agentcore:us-east-1:790402872574:runtime/travel_companion_basic-hYRzmCFvqR
ECR URI: 790402872574.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-travel_companion_basic


In [17]:
# Check deployment status
import time

print("⏳ Checking deployment status...")
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

while status not in end_status:
    print(f"Status: {status}")
    time.sleep(30)  # Check every 30 seconds
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']

print(f"\n🎉 Final Status: {status}")

if status == 'READY':
    print("✅ Agent successfully deployed to AgentCore Runtime!")
else:
    print("❌ Deployment failed. Check AWS console for details.")

# Return to original directory
os.chdir(original_dir)

⏳ Checking deployment status...


Retrieved Bedrock AgentCore status for: travel_companion_basic



🎉 Final Status: READY
✅ Agent successfully deployed to AgentCore Runtime!


## Step 7: Test Deployed Agent

In [18]:
# Test the deployed agent
if status == 'READY':
    print("🧪 Testing deployed Travel Agent")
    print("=" * 50)
    
    # Test basic interaction
    test_payload = {"prompt": "Hi, I want to plan a 10-day trip to Italy with a $5000 budget"}
    invoke_response = agentcore_runtime.invoke(test_payload)
    
    print(f"User: {test_payload['prompt']}")
    print(f"Agent: {invoke_response['response'][0]}")
else:
    print("⚠️ Cannot test - deployment not ready")

🧪 Testing deployed Travel Agent
User: Hi, I want to plan a 10-day trip to Italy with a $5000 budget
Agent: Great! Based on your budget and preferences, here's a breakdown of your $5000 budget for 10 days in Italy:

- Daily budget: $500 per day
- Flights: ~$1,200
- Accommodation: ~$1,800 (aligns with your mid-range hotel preference)
- Food: ~$1,000 (I'll keep in mind your vegetarian preference)
- Activities & Sightseeing: ~$800
- Buffer/Emergency: ~$200

To help you plan further, I'd like to know which cities in Italy you're interested in visiting. Italy has many wonderful destinations like Rome, Florence, Venice, Amalfi Coast, Milan, and more.

Could you share which specific cities or regions you'd like to include in your 10-day itinerary? This will help me provide more tailored recommendations for accommodations, transportation between cities, and must-see attractions that align with your budget.


In [19]:
# Test budget calculation functionality
if status == 'READY':
    test_payload = {"prompt": "Can you help me allocate my $5000 budget for 10 days?"}
    invoke_response = agentcore_runtime.invoke(test_payload)
    
    print(f"\nUser: {test_payload['prompt']}")
    print(f"Agent: {invoke_response['response'][0]}")


User: Can you help me allocate my $5000 budget for 10 days?
Agent: "Here's a detailed breakdown of how your $5000 budget can be allocated for your 10-day Italian adventure:\n\n### Budget Allocation\n- **Daily Budget**: $500 per day\n- **Flights**: $1,200 (international flights to/from Italy)\n- **Hotels**: $1,800 (about $180 per night)\n- **Food**: $1,000 (approximately $100 per day)\n- **Activities & Sightseeing**: $800 (about $80 per day for museums, tours, etc.)\n- **Emergency Buffer**: $200 (for unexpected expenses)\n\n### Tips for Making the Most of Your Budget\n\n**Flights ($1,200)**\n- Consider booking 2-3 months in advance for better rates\n- Be flexible with dates if possible (midweek flights are often cheaper)\n- Look for flights to major hubs like Rome or Milan\n\n**Accommodation ($1,800)**\n- This budget allows for mid-range hotels or nice Airbnbs\n- Stay in centrally located places to save on transportation\n- Consider staying in B&Bs which often include breakfast\n\n**Fo

## Step 8: Multi-turn Conversation Test

In [20]:
# Test multi-turn conversation
if status == 'READY':
    print("🗣️ Testing Multi-turn Conversation")
    print("=" * 50)
    
    conversation = [
        "I want to visit Italy",
        "I prefer mid-range hotels and vegetarian food",
        "Tell me about Rome's attractions",
        "What about Florence?"
    ]
    
    for i, message in enumerate(conversation, 1):
        print(f"\n--- Turn {i} ---")
        test_payload = {"prompt": message}
        invoke_response = agentcore_runtime.invoke(test_payload)
        
        print(f"User: {message}")
        print(f"Agent: {invoke_response['response'][0][:200]}...")  # Truncate for readability
        

🗣️ Testing Multi-turn Conversation

--- Turn 1 ---
User: I want to visit Italy
Agent: "Based on your 10-day timeframe and $5000 budget, here are some suggestions for planning your Italian adventure:\n\n### Popular Destinations to Consider:\n\n**Rome** \n- Currency: Euro (EUR)\n- Langua...

--- Turn 2 ---
User: I prefer mid-range hotels and vegetarian food
Agent: "Based on your preferences and budget, here's how you can make the most of your Italian vacation:\n\n### Accommodation: Mid-Range Hotels ($1,800 total, ~$180/night)\n- Look for 3-4 star hotels in cent...

--- Turn 3 ---
User: Tell me about Rome's attractions
Agent: "# Rome's Top Attractions\n\nRome is an incredible city filled with history, art, and culture at every turn. Here's a detailed guide to Rome's major attractions and some hidden gems that would fit wel...

--- Turn 4 ---
User: What about Florence?
Agent: "# Florence's Top Attractions\n\nFlorence (Firenze) is the birthplace of the Renaissance and home to some of the wo

## Step 9: Save Runtime Information

In [22]:
# Save runtime information for use in subsequent notebooks
import json

if status == 'READY':
    runtime_info = {
        "agent_name": agent_name,
        "agent_arn": launch_result.agent_arn,
        "agent_id": launch_result.agent_id,
        "ecr_uri": launch_result.ecr_uri,
        "region": region,
        "status": status
    }
    
    # Save to file for next notebooks
    with open('environments/runtime_info.json', 'w') as f:
        json.dump(runtime_info, f, indent=2)
    
    print("💾 Runtime information saved to environments/runtime_info.json")
    print("\n📋 Runtime Summary:")
    for key, value in runtime_info.items():
        print(f"  {key}: {value}")
else:
    print("⚠️ Runtime not ready - information not saved")

💾 Runtime information saved to environments/runtime_info.json

📋 Runtime Summary:
  agent_name: travel_companion_basic
  agent_arn: arn:aws:bedrock-agentcore:us-east-1:790402872574:runtime/travel_companion_basic-hYRzmCFvqR
  agent_id: travel_companion_basic-hYRzmCFvqR
  ecr_uri: 790402872574.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-travel_companion_basic
  region: us-east-1
  status: READY


## Next Steps

✅ **Completed in this notebook:**
- AgentCore Runtime configuration and setup
- Basic travel agent with Strands and Bedrock
- Multi-turn conversation capabilities
- Production deployment to AWS

➡️ **Next: Notebook 03 - Gateway Integration**
- Integrate external APIs (flights, hotels, weather, currency)
- Create OpenAPI 3.0 specifications
- Set up MCP Gateway with OAuth
- Test real API integrations